# Notebook 7: Citation Counts
We compare the total accumulated citations from publication to the last year with full citation data (2025). focusing on the difference between *sharing* and *non-sharing* articles, with a drill-down to the sharing subcategories (granularities).
<br><br>
Our analyses have two steps: First, we use a uni-variate approach (t-test or Mann-Whitney U test, depending on normality). Second, we fit a multivariable regression to test whether the findings hold when controlling for other known predictors of article impact (e.g., publication age, number of authors, open-access status, etc.).
<br><br>
_Note: the bottom section of this notebook stores a now-depricated analysis of citation residuals, which was used in the v1 manuscript but has been replaced by the multivariable regression above. It is kept here for reproducibility of the v1 results._

In [1]:
import numpy as np
import pandas as pd
import pingouin as pg
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.outliers_influence import variance_inflation_factor
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

from helpers.config import *
from helpers import dataset
from helpers.plotting import save_figure
from helpers.stats import compare_continuous

pio.renderers.default = "browser"

## Setup

In [2]:
combined, FEATURES_DF = dataset.load_or_build()
CITATIONS_DF = dataset.build_citations_frame(combined, FEATURES_DF)

CITATIONS_DF.columns

Loaded cached dataset from C:\Users\nirjo\Documents\University\PhD\Projects\eye_movement_data_sharing\data_store (234 rows)


Index(['is_sharing_data', 'sharing_class', 'has_us_author', 'is_open_access',
       'has_preprint', 'log_weeks_since_pub', 'log_number_of_authors',
       'venue_impact', 'venue_name', 'log_citations'],
      dtype='str')

## Sharing vs. Non-Sharing

In [3]:
DV = "log_citations"
GROUPING_VARIABLE = "is_sharing_data"

### Univariate Comparison

In [4]:
# compare summary statistics of citation counts between sharing and non-sharing articles
CITATIONS_DF.groupby(GROUPING_VARIABLE)[DV].describe().T

is_sharing_data,0,1
count,154.000000,80.000000
mean,2.656499,2.707260
std,0.889737,0.938678
min,0.000000,0.693147
25%,2.079442,2.197225
50%,2.639057,2.673554
75%,3.295837,3.433987
max,4.700480,4.762174


In [5]:
# check for normality
sw_share = stats.shapiro(CITATIONS_DF[CITATIONS_DF[GROUPING_VARIABLE] == 1][DV])
print(f"Sharing Group is Normal:\t\t{'YES' if sw_share.pvalue > 0.05 else 'NO'} (W={sw_share.statistic :.3f}; p={sw_share.pvalue :.3f})")

sw_non_share = stats.shapiro(CITATIONS_DF[CITATIONS_DF[GROUPING_VARIABLE] == 0][DV])
print(f"Non-Sharing Group is Normal:\t{'YES' if sw_non_share.pvalue > 0.05 else 'NO'} (W={sw_non_share.statistic :.3f}; p={sw_non_share.pvalue :.3f})")

if sw_share.pvalue > 0.05 and sw_non_share.pvalue > 0.05:
    print("Both groups are normally distributed, using t-test.")
else:
    print("At least one group is not normally distributed, using Mann-Whitney U test.")

Sharing Group is Normal:		YES (W=0.980; p=0.228)
Non-Sharing Group is Normal:	YES (W=0.988; p=0.223)
Both groups are normally distributed, using t-test.


In [6]:
# run univariate test
citations_univariate_test_results, citations_univariate_summary = compare_continuous(
    data=CITATIONS_DF,
    share_feature=GROUPING_VARIABLE,
    tested_feature=DV,
    alternative="greater",
    verbose=False
)
citations_univariate_test_results = pd.Series(citations_univariate_test_results).rename(DV)
is_significant = citations_univariate_test_results.loc["p_val"] < 0.05

print(f"Univariate test is {'' if is_significant else 'NOT '}significant ({citations_univariate_test_results.loc['test']}={citations_univariate_test_results.loc['statistic']:.3f}, p={citations_univariate_test_results.loc['p_val']:.4f})")

citations_univariate_test_results

Univariate test is NOT significant (t=0.399, p=0.3451)


test                                                            t
alternative                                               greater
statistic                                                 0.39938
p_val                                                    0.345086
dof                                                     152.76833
effect_sizes    {'Cohen d': 0.05598459193658729, 'CLES': 0.515...
Name: log_citations, dtype: object

### Multivariable Regression
We compare *sharing* vs. *non-sharing* articles while controlling for other known predictors of citation impact. We use three differnt models, varying in the way they account for the venue of publication:
- The first model is a simple OLS regression, using the venue impact score as a covariate.
- The second model is a mixed-effects model, with venue as a random effect.
- The third model merges the two approaches, using a LME with venue impact as a fixed effect and venue as a random effect. This is the most conservative model, and the one we report in the manuscript.

#### Model 1 :: OLS
Model 1 uses venue impact scores as a covariate, and does not account for the venue of publication as a random effect. The model is specified as follows:
$$ log(1 + \text{Raw Citation Counts}) \sim 1 + C(\text{Sharing Status}) + C(\text{Is OA}) + C(\text{Has US Author}) + C(\text{Has Preprint}) + log(\text{Weeks Since Pub.}) + log(\text{Number of Authors}) + \text{Impact Score} $$

In [7]:
model1 = smf.ols(
    f"{DV} ~ C({GROUPING_VARIABLE}) + C(has_us_author) + C(is_open_access) + C(has_preprint) + log_weeks_since_pub + log_number_of_authors + venue_impact",
    data=CITATIONS_DF,
    drop_cols=["sharing_class"]
).fit()

model1.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:          log_citations   R-squared:                       0.166
Model:                            OLS   Adj. R-squared:                  0.140
Method:                 Least Squares   F-statistic:                     6.416
Date:                Fri, 09 Oct 2026   Prob (F-statistic):           6.76e-07
Time:                        15:53:47   Log-Likelihood:                -286.98
No. Observations:                 234   AIC:                             590.0
Df Residuals:                     226   BIC:                             617.6
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
===========================================================================================
                              coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                  -4.6207      1.195     -3.866      0.000      -6.976      -2.265
C(is_sharing_data)[T.1]     0.0752      0.127      0.592      0.555      -0.175       0.325
C(has_us_author)[T.1]       0.1801      0.112      1.612      0.108      -0.040       0.400
C(is_open_access)[T.1]      0.3515      0.157      2.232      0.027       0.041       0.662
C(has_preprint)[T.1]        0.1657      0.144      1.151      0.251      -0.118       0.449
log_weeks_since_pub         1.1266      0.199      5.675      0.000       0.735       1.518
log_number_of_authors       0.1066      0.132      0.810      0.419      -0.153       0.366
venue_impact                0.1099      0.045      2.443      0.015       0.021       0.199
==============================================================================
Omnibus:                        4.298   Durbin-Watson:                   2.072
Prob(Omnibus):                  0.117   Jarque-Bera (JB):                4.040
Skew:                          -0.317   Prob(JB):                        0.133
Kurtosis:                       3.108   Cond. No.                         144.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

#### Model 2 :: Mixed-Effects
Model 2 uses venue as a random effect, and does not include the venue impact score as a covariate. The model is specified as follows:
$$ log(1 + \text{Raw Citation Counts}) \sim 1 + C(\text{Sharing Status}) + C(\text{Is OA}) + C(\text{Has US Author}) + C(\text{Has Preprint}) + log(\text{Weeks Since Pub.}) + log(\text{Number of Authors}) + (1 | \text{Venue}) $$

In [8]:
model2 = smf.mixedlm(
    f"{DV} ~ C({GROUPING_VARIABLE}) + C(has_us_author) + C(is_open_access) + C(has_preprint) +  log_weeks_since_pub + log_number_of_authors",
    data=CITATIONS_DF,
    groups=CITATIONS_DF["venue_name"],
    drop_cols=["sharing_class"]
).fit(reml=False)

model2.summary()

<class 'statsmodels.iolib.summary2.Summary'>
"""
              Mixed Linear Model Regression Results
==================================================================
Model:               MixedLM   Dependent Variable:   log_citations
No. Observations:    234       Method:               ML           
No. Groups:          23        Scale:                0.6494       
Min. group size:     1         Log-Likelihood:       -287.4621    
Max. group size:     43        Converged:            Yes          
Mean group size:     10.2                                         
------------------------------------------------------------------
                        Coef.  Std.Err.   z    P>|z| [0.025 0.975]
------------------------------------------------------------------
Intercept               -4.347    1.173 -3.704 0.000 -6.647 -2.047
C(is_sharing_data)[T.1]  0.092    0.131  0.705 0.481 -0.164  0.349
C(has_us_author)[T.1]    0.162    0.110  1.470 0.142 -0.054  0.377
C(is_open_access)[T.1]   0.339    0.173  1.957 0.050 -0.000  0.678
C(has_preprint)[T.1]     0.110    0.143  0.773 0.439 -0.169  0.390
log_weeks_since_pub      1.120    0.196  5.718 0.000  0.736  1.504
log_number_of_authors    0.149    0.130  1.153 0.249 -0.105  0.403
Group Var                0.054    0.057                           
==================================================================

"""

#### Model 3 :: Mixed-Effects with Venue Impact
Model 3 uses venue as a random effect, and includes the venue impact score as a covariate. The model is specified as follows:
$$ log(1 + \text{Raw Citation Counts}) \sim 1 + C(\text{Sharing Status}) + C(\text{Is OA}) + C(\text{Has US Author}) + C(\text{Has Preprint}) + log(\text{Weeks Since Pub.}) + log(\text{Number of Authors}) + \text{Impact Score} + (1 | \text{Venue}) $$

In [9]:
model3 = smf.mixedlm(
    f"{DV} ~ C({GROUPING_VARIABLE}) + C(has_us_author) + C(is_open_access) + C(has_preprint) +  log_weeks_since_pub + log_number_of_authors + venue_impact",
    data=CITATIONS_DF,
    groups=CITATIONS_DF["venue_name"],
    drop_cols=["sharing_class"]
).fit(reml=False)

model3.summary()

<class 'statsmodels.iolib.summary2.Summary'>
"""
              Mixed Linear Model Regression Results
==================================================================
Model:               MixedLM   Dependent Variable:   log_citations
No. Observations:    234       Method:               ML           
No. Groups:          23        Scale:                0.6294       
Min. group size:     1         Log-Likelihood:       -283.8143    
Max. group size:     43        Converged:            Yes          
Mean group size:     10.2                                         
------------------------------------------------------------------
                        Coef.  Std.Err.   z    P>|z| [0.025 0.975]
------------------------------------------------------------------
Intercept               -4.688    1.162 -4.033 0.000 -6.966 -2.410
C(is_sharing_data)[T.1]  0.044    0.129  0.340 0.734 -0.208  0.296
C(has_us_author)[T.1]    0.142    0.108  1.317 0.188 -0.070  0.354
C(is_open_access)[T.1]   0.331    0.171  1.937 0.053 -0.004  0.665
C(has_preprint)[T.1]     0.102    0.140  0.731 0.465 -0.172  0.376
log_weeks_since_pub      1.118    0.193  5.801 0.000  0.740  1.496
log_number_of_authors    0.112    0.128  0.871 0.384 -0.140  0.363
venue_impact             0.170    0.068  2.503 0.012  0.037  0.303
Group Var                0.053    0.050                           
==================================================================

"""

### Model Comparison
We have results from the 3 models above, and we can compare them to see how the inclusion of venue as a random effect and/or the venue impact score as a covariate affects the model convergence and predictive power.<br>
We also check if the conclusion - the estimated effect of sharing on citation counts - is consistent across the three models.

In [10]:
MODELS = {"1: OLS": model1, "2: LME (venue)": model2, "3: LME (venue + impact)": model3}

In [11]:
def model_diagnostics(result) -> dict:
    if not isinstance(result.model, sm.MixedLM):
        return {"converged": "n/a", "N": int(result.nobs), "R2 marginal": result.rsquared, "R2 conditional": result.rsquared, "ICC": np.nan}
    marginal, conditional = _nakagawa_r2(result)
    venue_var = result.cov_re.iloc[0, 0]
    return {
        "converged": result.converged, "N": int(result.nobs),
        "R2 marginal": marginal, "R2 conditional": conditional,
        "ICC": venue_var / (venue_var + result.scale),
    }


def sharing_effect(result) -> dict:
    GROUPING_TERM = f"C({GROUPING_VARIABLE})[T.1]"
    ci_low, ci_high = result.conf_int().loc[GROUPING_TERM]
    return {
        "b": result.params[GROUPING_TERM], "SE": result.bse[GROUPING_TERM], "p": result.pvalues[GROUPING_TERM],
        "CI low": ci_low, "CI high": ci_high,
        "% diff": _percent_difference(result.params[GROUPING_TERM]),
        "% diff CI low": _percent_difference(ci_low), "% diff CI high": _percent_difference(ci_high),
    }


def _nakagawa_r2(result) -> tuple[float, float]:
    """Marginal (fixed effects only) and conditional (fixed effects + venue) R^2 of a `MixedLM` fit."""
    fixed_var = np.var(result.model.exog @ result.fe_params.values)
    venue_var = result.cov_re.iloc[0, 0]
    total_var = fixed_var + venue_var + result.scale
    return fixed_var / total_var, (fixed_var + venue_var) / total_var


def _percent_difference(b: float) -> float:
    """The DV is log(1 + citations), so exp(b) - 1 is the proportional difference in citations."""
    return 100 * (np.exp(b) - 1)

In [12]:
# compare model diagnostics:
diagnostics = pd.DataFrame({name: model_diagnostics(res) for name, res in MODELS.items()}).T
diagnostics.index.name = "Model"

print(f"Best r^2 (marginal) is {diagnostics['R2 marginal'].max():.3f} in model {diagnostics['R2 marginal'].idxmax()}")
print(f"Best r^2 (conditional) is {diagnostics['R2 conditional'].max():.3f} in model {diagnostics['R2 conditional'].idxmax()}")
diagnostics

Best r^2 (marginal) is 0.185 in model 3: LME (venue + impact)
Best r^2 (conditional) is 0.248 in model 3: LME (venue + impact)


,converged,N,R2 marginal,R2 conditional,ICC
Model,,,,,
1: OLS,n/a,234,0.165779,0.165779,NaN
2: LME (venue),True,234,0.135879,0.202205,0.076755
3: LME (venue + impact),True,234,0.18522,0.248053,0.077117


In [13]:
# compare sharing effects across models
sharing_effects = pd.DataFrame({name: sharing_effect(res) for name, res in MODELS.items()}).T

same_sign = sharing_effects["b"].gt(0).all() or sharing_effects["b"].lt(0).all()
ci_includes_zero = (sharing_effects["CI low"] < 0).all() and (sharing_effects["CI high"] > 0).all()
print(f"Sharing coefficient ranges from {sharing_effects['b'].min():.3f} to {sharing_effects['b'].max():.3f} across models")
print(f"Same sign in all three models: {'YES' if same_sign else 'NO'}")
print(f"95% CI includes 0 in all three models: {'YES' if ci_includes_zero else 'NO'}")

sharing_effects

Sharing coefficient ranges from 0.044 to 0.092 across models
Same sign in all three models: YES
95% CI includes 0 in all three models: YES


,b,SE,p,CI low,CI high,% diff,% diff CI low,% diff CI high
1: OLS,0.075154,0.127001,0.554600,-0.175102,0.325411,7.805052,-16.062880,38.459947
2: LME (venue),0.092229,0.130762,0.480610,-0.164059,0.348517,9.661576,-15.130803,41.696419
3: LME (venue + impact),0.043755,0.128570,0.733613,-0.208237,0.295748,4.472671,-18.798576,34.413147


---

## Sharing Granularity

In [14]:
GROUPING_VARIABLE = "sharing_class"

CITATIONS_GRANULARITY_DF = CITATIONS_DF.loc[CITATIONS_DF[GROUPING_VARIABLE] != "NONE"]
CITATIONS_GRANULARITY_DF[GROUPING_VARIABLE] = CITATIONS_GRANULARITY_DF[GROUPING_VARIABLE].cat.remove_unused_categories()

# verify we are left only with the three sharing classes
CITATIONS_GRANULARITY_DF[GROUPING_VARIABLE].unique()

['FIXATION', 'TRIAL', 'PARTICIPANT']
Categories (3, str): ['PARTICIPANT' < 'TRIAL' < 'FIXATION']

### Univariate Comparison

In [15]:
# compare summary statistics of citation counts between sharing granularities
CITATIONS_GRANULARITY_DF.groupby(GROUPING_VARIABLE)[DV].describe().T

sharing_class,PARTICIPANT,TRIAL,FIXATION
count,23.000000,26.000000,31.000000
mean,2.714876,2.472695,2.898341
std,0.995560,0.927505,0.889432
min,0.693147,0.693147,1.098612
25%,2.191013,2.108887,2.302585
50%,2.995732,2.302585,2.944439
75%,3.312696,3.044522,3.637586
max,4.762174,4.127134,4.304065


In [16]:
all_normal = True
for sg in CITATIONS_GRANULARITY_DF[GROUPING_VARIABLE].unique():
    # check for normality
    sw = stats.shapiro(
        CITATIONS_GRANULARITY_DF[CITATIONS_GRANULARITY_DF[GROUPING_VARIABLE] == sg][DV]
    )
    sw_p = sw.pvalue
    print(f"{sg} Sharing Group is Normal:\t{'YES' if sw_p > 0.05 else 'NO'} (W={sw.statistic :.3f}; p={sw_p :.3f})")
    all_normal &= sw_p > 0.05

print()
print(f"All sharing groups are normally distributed: {'YES' if all_normal else 'NO'}")
print(f"Using {'ANOVA' if all_normal else 'Kruskal-Wallis'} test for group comparisons.")

FIXATION Sharing Group is Normal:	YES (W=0.960; p=0.285)
TRIAL Sharing Group is Normal:	YES (W=0.960; p=0.391)
PARTICIPANT Sharing Group is Normal:	YES (W=0.929; p=0.104)

All sharing groups are normally distributed: YES
Using ANOVA test for group comparisons.


In [20]:
# run univariate test
citations_granularity_results, _citations_granularity_summary = compare_continuous(
    data=CITATIONS_GRANULARITY_DF,
    share_feature=GROUPING_VARIABLE,
    tested_feature=DV,
    alternative="greater",
    verbose=False
)
citations_granularity_results = pd.Series(citations_granularity_results).rename(DV)
is_significant = citations_granularity_results.loc["p_val"] < 0.05

print(f"Univariate test is {'' if is_significant else 'NOT '}significant ({citations_granularity_results.loc['test']}={citations_granularity_results.loc['statistic']:.3f}, p={citations_granularity_results.loc['p_val']:.4f})")

citations_granularity_results

Univariate test is NOT significant (ANOVA=1.472, p=0.2358)


test                                    ANOVA
statistic                            1.472223
p_val                                  0.2358
dof                                   (2, 77)
effect_sizes    {'eta2': 0.03683115523541124}
Name: log_citations, dtype: object

### Multivariable Regression
We compare the three sharing granularities while controlling for other known predictors of citation impact.<br>
We use the same three models as above, varying in the way they account for the venue of publication:

#### Model 1 :: OLS
$$ log(1 + \text{Raw Citation Counts}) \sim 1 + C(\text{Sharing Granularity}) + C(\text{Is OA}) + C(\text{Has US Author}) + C(\text{Has Preprint}) + log(\text{Weeks Since Pub.}) + log(\text{Number of Authors}) + \text{Impact Score} $$

#### Model 2 :: Mixed-Effects
$$ log(1 + \text{Raw Citation Counts}) \sim 1 + C(\text{Sharing Granularity}) + C(\text{Is OA}) + C(\text{Has US Author}) + C(\text{Has Preprint}) + log(\text{Weeks Since Pub.}) + log(\text{Number of Authors}) + (1 | \text{Venue}) $$

#### Model 3 :: Mixed-Effects with Venue Impact
$$ log(1 + \text{Raw Citation Counts}) \sim 1 + C(\text{Sharing Granularity}) + C(\text{Is OA}) + C(\text{Has US Author}) + C(\text{Has Preprint}) + log(\text{Weeks Since Pub.}) + log(\text{Number of Authors}) + \text{Impact Score} + (1 | \text{Venue}) $$

In [22]:
gran_model1 = smf.ols(
    f"{DV} ~ C({GROUPING_VARIABLE}) + C(has_us_author) + C(is_open_access) + C(has_preprint) + log_weeks_since_pub + log_number_of_authors + venue_impact",
    data=CITATIONS_DF,
    drop_cols=["is_sharing_data"]
).fit()

gran_model1.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:          log_citations   R-squared:                       0.168
Model:                            OLS   Adj. R-squared:                  0.135
Method:                 Least Squares   F-statistic:                     5.031
Date:                Fri, 09 Oct 2026   Prob (F-statistic):           3.47e-06
Time:                        16:01:36   Log-Likelihood:                -286.65
No. Observations:                 234   AIC:                             593.3
Df Residuals:                     224   BIC:                             627.9
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
===================================================================================================
                                      coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
Intercept                          -4.4490      1.220     -3.646      0.000      -6.853      -2.044
C(sharing_class)[T.PARTICIPANT]     0.0420      0.196      0.215      0.830      -0.343       0.427
C(sharing_class)[T.TRIAL]          -0.0109      0.195     -0.056      0.956      -0.395       0.373
C(sharing_class)[T.FIXATION]        0.1684      0.177      0.951      0.343      -0.181       0.518
C(has_us_author)[T.1]               0.1755      0.115      1.527      0.128      -0.051       0.402
C(is_open_access)[T.1]              0.3399      0.159      2.137      0.034       0.026       0.653
C(has_preprint)[T.1]                0.1518      0.147      1.031      0.304      -0.138       0.442
log_weeks_since_pub                 1.1018      0.202      5.464      0.000       0.704       1.499
log_number_of_authors               0.0878      0.134      0.653      0.514      -0.177       0.353
venue_impact                        0.1132      0.045      2.496      0.013       0.024       0.203
==============================================================================
Omnibus:                        4.324   Durbin-Watson:                   2.060
Prob(Omnibus):                  0.115   Jarque-Bera (JB):                4.033
Skew:                          -0.314   Prob(JB):                        0.133
Kurtosis:                       3.136   Cond. No.                         146.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [23]:
gran_model2 = smf.mixedlm(
    f"{DV} ~ C({GROUPING_VARIABLE}) + C(has_us_author) + C(is_open_access) + C(has_preprint) +  log_weeks_since_pub + log_number_of_authors",
    data=CITATIONS_DF,
    groups=CITATIONS_DF["venue_name"],
    drop_cols=["is_sharing_data"]
).fit(reml=False)

gran_model2.summary()

<class 'statsmodels.iolib.summary2.Summary'>
"""
                  Mixed Linear Model Regression Results
==========================================================================
Model:                 MixedLM      Dependent Variable:      log_citations
No. Observations:      234          Method:                  ML           
No. Groups:            23           Scale:                   0.6462       
Min. group size:       1            Log-Likelihood:          -287.1967    
Max. group size:       43           Converged:               Yes          
Mean group size:       10.2                                               
--------------------------------------------------------------------------
                                Coef.  Std.Err.   z    P>|z| [0.025 0.975]
--------------------------------------------------------------------------
Intercept                       -4.178    1.194 -3.498 0.000 -6.519 -1.837
C(sharing_class)[T.PARTICIPANT]  0.090    0.200  0.451 0.652 -0.302  0.483
C(sharing_class)[T.TRIAL]       -0.006    0.197 -0.031 0.975 -0.391  0.379
C(sharing_class)[T.FIXATION]     0.161    0.174  0.924 0.356 -0.180  0.501
C(has_us_author)[T.1]            0.152    0.113  1.342 0.180 -0.070  0.373
C(is_open_access)[T.1]           0.330    0.174  1.898 0.058 -0.011  0.671
C(has_preprint)[T.1]             0.102    0.145  0.707 0.479 -0.181  0.386
log_weeks_since_pub              1.096    0.198  5.524 0.000  0.707  1.485
log_number_of_authors            0.134    0.131  1.027 0.304 -0.122  0.391
Group Var                        0.058    0.061                           
==========================================================================

"""

In [24]:
gran_model3 = smf.mixedlm(
    f"{DV} ~ C({GROUPING_VARIABLE}) + C(has_us_author) + C(is_open_access) + C(has_preprint) +  log_weeks_since_pub + log_number_of_authors + venue_impact",
    data=CITATIONS_DF,
    groups=CITATIONS_DF["venue_name"],
    drop_cols=["is_sharing_data"]
).fit(reml=False)

gran_model3.summary()

<class 'statsmodels.iolib.summary2.Summary'>
"""
                  Mixed Linear Model Regression Results
==========================================================================
Model:                 MixedLM      Dependent Variable:      log_citations
No. Observations:      234          Method:                  ML           
No. Groups:            23           Scale:                   0.6264       
Min. group size:       1            Log-Likelihood:          -283.4697    
Max. group size:       43           Converged:               Yes          
Mean group size:       10.2                                               
--------------------------------------------------------------------------
                                Coef.  Std.Err.   z    P>|z| [0.025 0.975]
--------------------------------------------------------------------------
Intercept                       -4.511    1.180 -3.822 0.000 -6.825 -2.198
C(sharing_class)[T.PARTICIPANT]  0.040    0.197  0.202 0.840 -0.346  0.425
C(sharing_class)[T.TRIAL]       -0.064    0.192 -0.334 0.739 -0.441  0.313
C(sharing_class)[T.FIXATION]     0.122    0.171  0.713 0.476 -0.214  0.458
C(has_us_author)[T.1]            0.132    0.111  1.189 0.234 -0.086  0.350
C(is_open_access)[T.1]           0.320    0.171  1.869 0.062 -0.016  0.656
C(has_preprint)[T.1]             0.094    0.142  0.662 0.508 -0.184  0.371
log_weeks_since_pub              1.092    0.195  5.597 0.000  0.709  1.474
log_number_of_authors            0.095    0.130  0.731 0.465 -0.159  0.349
venue_impact                     0.173    0.069  2.527 0.011  0.039  0.308
Group Var                        0.055    0.053                           
==========================================================================

"""

### Model Comparison

In [25]:
GRAN_MODELS = {"1: OLS": gran_model1, "2: LME (venue)": gran_model2, "3: LME (venue + impact)": gran_model3}

In [26]:
# compare model diagnostics:
gran_diagnostics = pd.DataFrame({name: model_diagnostics(res) for name, res in GRAN_MODELS.items()}).T
gran_diagnostics.index.name = "Model"

print(f"Best r^2 (marginal) is {gran_diagnostics['R2 marginal'].max():.3f} in model {gran_diagnostics['R2 marginal'].idxmax()}")
print(f"Best r^2 (conditional) is {gran_diagnostics['R2 conditional'].max():.3f} in model {gran_diagnostics['R2 conditional'].idxmax()}")
gran_diagnostics

Best r^2 (marginal) is 0.188 in model 3: LME (venue + impact)
Best r^2 (conditional) is 0.253 in model 3: LME (venue + impact)


,converged,N,R2 marginal,R2 conditional,ICC
Model,,,,,
1: OLS,n/a,234,0.168138,0.168138,NaN
2: LME (venue),True,234,0.137484,0.208453,0.082282
3: LME (venue + impact),True,234,0.187667,0.253265,0.080752


In [27]:
# compare sharing effects across models
gran_sharing_effects = pd.DataFrame({name: sharing_effect(res) for name, res in GRAN_MODELS.items()}).T

same_sign = gran_sharing_effects["b"].gt(0).all() or gran_sharing_effects["b"].lt(0).all()
ci_includes_zero = (gran_sharing_effects["CI low"] < 0).all() and (gran_sharing_effects["CI high"] > 0).all()
print(f"Sharing coefficient ranges from {gran_sharing_effects['b'].min():.3f} to {gran_sharing_effects['b'].max():.3f} across models")
print(f"Same sign in all three models: {'YES' if same_sign else 'NO'}")
print(f"95% CI includes 0 in all three models: {'YES' if ci_includes_zero else 'NO'}")

gran_sharing_effects

KeyError: 'C(sharing_class)[T.1]'

In [17]:
1/0 # keep here to avoid running unnecessary code

ZeroDivisionError: division by zero

### Sharing-Class Comparisons
We repeat the same two-step analysis for the three sharing classes (FIXATION, TRIAL, PARTICIPANT) to determine whether there are differences in citations between the different levels of sharing granularity.

In [ ]:
# remove level "NONE" from the dataset
CITATIONS_GRANULARITY_DF = CITATIONS_DF.loc[CITATIONS_DF["sharing_class"] != "NONE"]

# reset the categorical axis with PARTICIPANT as baseline
CITATIONS_GRANULARITY_DF["sharing_class"] = CITATIONS_GRANULARITY_DF["sharing_class"].cat.remove_unused_categories()

# compare the levels
CITATIONS_GRANULARITY_DF.groupby("sharing_class")["log_citations"].describe()

In [ ]:
citations_granularity_results, _ = compare_continuous(
    data=CITATIONS_GRANULARITY_DF, share_feature="sharing_class", tested_feature="log_citations", alternative="two-sided", verbose=False
)
citations_granularity_results = pd.Series(citations_granularity_results).rename("log_citations")
citations_granularity_results

In [ ]:
citations_granularity_model = smf.ols(
    "log_citations ~ C(sharing_class) + C(has_us_author) + C(is_open_access) + C(has_preprint) + venue_impact + log_weeks_since_pub + log_number_of_authors",
    data=CITATIONS_GRANULARITY_DF,
    drop_cols=["is_sharing_data"]
).fit()
contrast_ftest = citations_granularity_model.f_test('C(sharing_class)[T.TRIAL] = C(sharing_class)[T.FIXATION] = 0')

print(f"Do 'TRIAL' and 'PARTICIPANT' vary: {'YES' if citations_granularity_model.pvalues.loc["C(sharing_class)[T.TRIAL]"] < 0.05 else 'NO'}:\tp_unc={citations_granularity_model.pvalues.loc["C(sharing_class)[T.TRIAL]"] :.4f}")
print(f"Do 'FIXATION' and 'PARTICIPANT' vary: {'YES' if citations_granularity_model.pvalues.loc["C(sharing_class)[T.FIXATION]"] < 0.05 else 'NO'}:\tp_unc={citations_granularity_model.pvalues.loc["C(sharing_class)[T.FIXATION]"] :.4f}")
print(f"Do 'TRIAL' and 'FIXATION' vary: {'YES' if contrast_ftest.pvalue < 0.05 else 'NO'}:\tp_unc={contrast_ftest.pvalue :.4f}")

citations_granularity_model.summary()

In [ ]:
# TODO: drop this to see the (deprecated) residuals analysis
1/0

---

## Age-Adjusted Citation Residuals (SUPERSEDED)

**This section is superseded and does not appear in the revised manuscript.** It is the analysis
reported in the submitted version, kept so the published v1 results stay reproducible. Everything
from this heading to the end of the notebook belongs to it, including the per-group regressions at
the bottom.

It regresses log-citations on log-age alone, then compares the residuals between sharing and
non-sharing articles. The multivariable model above replaces it, for two reasons. Age is only one
of several predictors of citation count, so residualizing on age alone leaves venue, preprint
status and the rest uncontrolled. And the two-stage form (fit, then test the residuals) gives the
right point estimate with the wrong standard errors, because the second stage treats fitted
residuals as though they were observed data.

These cells deliberately use `Pub2UpdateTime` for article age rather than
`weeks_through_shared_year`. That is what v1 was computed from, and changing it here would stop
this section reproducing the published numbers, which is its only remaining purpose.

### Citation Counts over Time
#### (1) Quantify the Benefit of Data Sharing
We want to quantify the citation benefit of data sharing over time.<br>
To do so we first fit a log-log regression model to all data points, predicting citation counts from time since publication: $$\log(\text{Citations} +1) = \beta_0 + \beta_1 \log(\text{Weeks Since Publication}) + \epsilon$$(we add $1$ to the citation count to avoid $\log(0)$)<br><br>
Then, we compute the residuals (i.e., the difference between observed and predicted citation counts) for each article, and compare the residuals between sharing and non-sharing articles and across types of data shared.

In [ ]:
# fit log-linear model to all data
x_log = np.log(combined["Pub2UpdateTime"] / pd.Timedelta(weeks=1))
y_log = np.log(combined['TotalCitations'] + 1)  # add 1 to avoid log(0)
slope, intercept = np.polyfit(x_log, y_log, deg=1)

# compute residuals
ylog_pred = intercept + slope * x_log
residuals = (y_log - ylog_pred).rename("citation_residuals")
combined_with_residuals = pd.concat([combined, residuals], axis=1)

# extract residual by share/not-share
non_sharing_residuals = combined_with_residuals.loc[
    ~combined_with_residuals["is_sharing_data"], "citation_residuals"
]
sharing_residuals = combined_with_residuals.loc[
    combined_with_residuals["is_sharing_data"], "citation_residuals"
]

In [ ]:
# check for normality of residuals
print("Are citation residuals normally distributed for non-sharing articles?")
residuals_non_sharing_shapiro = stats.shapiro(non_sharing_residuals)
print(f"{"YES" if residuals_non_sharing_shapiro.pvalue > 0.05 else "NO"} (W={residuals_non_sharing_shapiro.statistic:.3f}, p={residuals_non_sharing_shapiro.pvalue:.4f})\n")

print("Are citation residuals normally distributed for sharing articles?")
residuals_sharing_shapiro = stats.shapiro(sharing_residuals)
print(f"{"YES" if residuals_sharing_shapiro.pvalue > 0.05 else "NO"} (W={residuals_sharing_shapiro.statistic:.3f}, p={residuals_sharing_shapiro.pvalue:.4f})\n")

# run Levene's test for homogeneity of variance
levene_test = stats.levene(non_sharing_residuals, sharing_residuals)
is_equal_var = levene_test.pvalue > 0.05
print(f"Levene's test for homogeneity of variance: W={levene_test.statistic:.3f}, p={levene_test.pvalue:.4f}")
print(f"Sample have {'EQUAL' if is_equal_var else 'UNEQUAL'} variances.")

In [ ]:
# run two-sample t-test
residuals_ttest = pg.ttest(
    non_sharing_residuals, sharing_residuals, paired=False, alternative="less", correction=~is_equal_var,
)
print("t-test:")
display(residuals_ttest)

# run Mann-Whitney U-test
residuals_mwtest = pg.mwu(non_sharing_residuals, sharing_residuals, alternative="less")
print("MW U-test:")
display(residuals_mwtest)

# calculate the percentage difference in mean citations
non_share_residuals_mean = non_sharing_residuals.mean()
share_residuals_mean = sharing_residuals.mean()
citation_ratio = np.exp(share_residuals_mean - non_share_residuals_mean)
percentage_diff = (citation_ratio - 1) * 100
print(f"Data-Sharing has {percentage_diff:.1f}% citations compared with Non-Sharing.")

#### Is there a Difference Between Types of Data Shared?

In [ ]:
share_group_residuals = []
for share_group in ["FIXATION", "TRIAL", "PARTICIPANT"]:
    group_residuals = (
        combined_with_residuals
        .loc[combined_with_residuals["data_sharing_class"] == share_group, "citation_residuals"]
        .dropna()
        .reset_index(drop=True)
        .rename(share_group)
    )
    share_group_residuals.append(group_residuals)
    shapiro_result = stats.shapiro(group_residuals)
    print(f"Are citation-residuals (in log-log space) normally distributed for {share_group} sharing articles?")
    print(f"{"YES" if shapiro_result.pvalue > 0.05 else "NO"} (W={shapiro_result.statistic:.3f}, p={shapiro_result.pvalue:.4f})\n")

print("Statistical Tests:")
# run one-way anova
print("One-Way ANOVA")
sharegroup_residuals_anova = stats.f_oneway(*share_group_residuals)
display(sharegroup_residuals_anova)

# run Kruskal-Wallis
print("Kruskal-Wallis")
sharegroup_residuals_kruskal = stats.kruskal(*share_group_residuals)
display(sharegroup_residuals_kruskal)

#### Visualizing Citation Residuals

In [ ]:
column_titles = ["Citations for Time Since Publication", "Citation Residuals by Data Sharing"]
residuals_fig = make_subplots(
    rows=1, cols=2, shared_xaxes=False, shared_yaxes=False,
    column_titles=column_titles, column_widths=[0.7, 0.3],
    horizontal_spacing=0.075,
)

# single-article data
for is_share in sorted(combined_with_residuals["is_sharing_data"].unique()):
    is_sharing_subset = combined_with_residuals[combined_with_residuals["is_sharing_data"] == is_share]
    legendgroup = "Sharing" if is_share else "Not Sharing"
    color = CLASS_COLORS[legendgroup.upper()]
    # left: scatter plot of citations over time
    for share_class in is_sharing_subset["data_sharing_class"].unique():
        label = share_class.title() if is_share else "Not Sharing"
        symbol = {"FIXATION": "circle", "TRIAL": "triangle-up", "PARTICIPANT": "square", "NONE": "x", }[share_class]
        class_subset = is_sharing_subset.loc[combined_with_residuals["data_sharing_class"] == share_class]
        residuals_fig.add_trace(
            row=1, col=1,
            trace=go.Scatter(
                name=label, legendgroup=label,
                x=class_subset["Pub2UpdateTime"] / pd.Timedelta(weeks=1),
                y=class_subset["TotalCitations"],
                mode="markers",
                marker=dict(color=CLASS_COLORS[share_class], symbol=symbol, size=8, ),
            )
        )
    residuals_fig.update_xaxes(
        row=1, col=1, title=dict(text="Weeks Since Publication", font=AXIS_TITLE_FONT, standoff=10), tickfont=AXIS_TICK_FONT,
    )
    residuals_fig.update_yaxes(
        row=1, col=1, title=dict(text="Total Citations", font=AXIS_TITLE_FONT, standoff=5), tickfont=AXIS_TICK_FONT,
    )

    # right: violin plot of residuals
    label = "Sharing" if is_share else "Not Sharing"
    residuals_fig.add_trace(
        row=1, col=2,
        trace=go.Violin(
            x0=0, y=is_sharing_subset["citation_residuals"],  # keeping it in log-space for visualization & statistics
            name=label, legendgroup=label, scalegroup=1,
            side='positive' if is_share else 'negative',
            fillcolor=color, line_color=color,
            width=0.95, spanmode="hard",
            points=False, pointpos=0, jitter=0.5,
            box=dict(visible=False, width=0.5, line=dict(color="black")),
            meanline=dict(visible=False, color='black'),
            showlegend=False,
        )
    )
    residuals_fig.update_xaxes(
        row=1, col=2, title=None, showticklabels=False,
    )
    residuals_fig.update_yaxes(
        row=1, col=2,
        title=dict(text="log-Residual Citations", font=AXIS_TITLE_FONT, standoff=5),
        tickfont=AXIS_TICK_FONT,
    )
    # add annotations
    residuals_fig.add_annotation(
        row=1, col=2,
        x=0.2 if is_share else -0.2,
        xanchor="left" if is_share else "right", xref="x",
        y=1.75, yanchor="top", yref="y2",
        text=f"{label.replace(" ", "<br>")}",
        font={**AXIS_TITLE_FONT, "color": color},
        showarrow=False,
    )
    residuals_fig.add_annotation(   # p value annotation
        row=1, col=2,
        x=0.125, xanchor="left", xref="x",
        y=-2.5, yanchor="middle", yref="y2",
        text=f"<i>p={residuals_ttest["p_val"].iloc[0] :.3f}</i>",
        font=AXIS_TITLE_FONT,
        showarrow=False,
    )

# add log-log trendline for citations over time
x_logspace = np.linspace(x_log.min(), x_log.max(), 100)
y_logpred = intercept + slope * x_logspace
y_pred = np.exp(y_logpred) - 1  # invert log transform
x_orig = np.exp(x_logspace)
residuals_fig.add_trace(
    row=1, col=1,
    trace=go.Scatter(
        name="Expected<br>Citations",
        x=x_orig, y=y_pred,
        mode="lines",
        line=dict(color='black', width=2.5, dash='dash'),
    )
)

# update layout and show
for ann in residuals_fig.layout.annotations:
    if ann.text not in column_titles:
        continue
    x = -0.03 if ann.text == column_titles[0] else 0.67
    ann.update(dict(
        font=AXIS_TITLE_FONT,
        x=x, xanchor="left", xref="paper",
        y=1.1, yanchor="top", yref="paper",
))
residuals_fig.update_xaxes(zeroline=False,)
residuals_fig.update_yaxes(
    showgrid=True, gridcolor='lightgrey', gridwidth=1.5,
    zeroline=False,
)
residuals_fig.update_xaxes(
    row=1, col=1, range=[140, 505]
)
residuals_fig.update_yaxes(
    row=1, col=1, range=[-5, 119]
)
residuals_fig.update_layout(
    width=1000, height=400,
    title=dict(
        text="<b>Estimating Number of Citations by Weeks Past Publication</b>",
        font=TITLE_FONT,
        x=0.5, xanchor="center", y=0.99, yanchor="top"
    ),
    violinmode='overlay',
    violingap=0,
    legend=dict(
        visible=True,
        orientation="v",
        entrywidth=30,
        bgcolor='rgba(255,255,255,0.2)',
        borderwidth=2, bordercolor='rgba(0,0,0,1)',
        x=0.01, xanchor="left", xref="paper",
        y=1.0, yanchor="top", yref="paper",
        font=LEGEND_FONT, font_size=10,
    ),
    margin=dict(t=60, b=5, l=50, r=10, pad=0),
    template="plotly_white"
)
residuals_fig.show()

In [ ]:
# flip to True to export this figure into `output/`
if False:
    save_figure(residuals_fig, "citation_residuals_by_data_sharing.png", width=1000, height=400)

#### (2) Regress each Sharing Group Separately
exploratory analysis: fit separate log-log regression models for each sharing group

In [ ]:
citation_counts_fig = go.Figure()
for group in combined["data_sharing_class"].unique():
    subset = combined[combined["data_sharing_class"] == group]
    color = CLASS_COLORS[group]
    symbol = 'circle' if group != "NONE" else 'x'
    citation_counts_fig.add_trace(go.Scatter(
        name=group, legendgroup=group,
        x=subset["Pub2UpdateTime"] / pd.Timedelta(weeks=1), y=subset["TotalCitations"],
        mode="markers", marker=dict(color=color, size=8, symbol=symbol),
    ))
    # fit log-log trendline per group
    x_log = np.log(subset["Pub2UpdateTime"] / pd.Timedelta(weeks=1))
    y_log = np.log(subset['TotalCitations'] + 1)  # add 1 to avoid log(0)
    slope, intercept = np.polyfit(x_log, y_log, deg=1)
    x_logspace = np.linspace(x_log.min(), x_log.max(), 100)
    y_logpred = intercept + slope * x_logspace
    y_pred = np.exp(y_logpred) - 1  # invert log transform
    x_orig = np.exp(x_logspace)
    citation_counts_fig.add_trace(go.Scatter(
        name=f"Trendline ({group})",
        x=x_orig, y=y_pred, mode="lines", line=dict(color=color, width=2, dash='dot'),
        showlegend=False,
    ))
    print("Group: {0: <12}".format(group) + f"Slope: {slope:.3f},\tIntercept: {intercept:.2f}")

# add general log-log trendline
x_log = np.log(combined["Pub2UpdateTime"] / pd.Timedelta(weeks=1))
y_log = np.log(combined['TotalCitations'] + 1)
slope, intercept = np.polyfit(x_log, y_log, deg=1)
x_logspace = np.linspace(x_log.min(), x_log.max(), 100)
y_logpred = intercept + slope * x_logspace
y_pred = np.exp(y_logpred) - 1
x_orig = np.exp(x_logspace)
citation_counts_fig.add_trace(go.Scatter(
    name="Trendline (all data)",
    x=x_orig, y=y_pred, mode="lines", line=dict(color='black', width=2, dash='dash'),
))
print("{0: <19}".format("All Data:") + f"Slope: {slope:.3f},\tIntercept: {intercept:.2f}")

citation_counts_fig.update_layout(
    width=800, height=300,
    title=dict(
        text="Citation Counts over Time Since Publication", font=TITLE_FONT,
        x=0.5, xanchor="center", y=0.95, yanchor="top",
    ),
    xaxis=dict(
        title=dict(text="Weeks Since Publication", font=AXIS_TITLE_FONT, standoff=10),
        tickfont=AXIS_TICK_FONT,
        # type="log",
        zeroline=False,
    ),
    yaxis=dict(
        title=dict(text="Total Citation Counts", font=AXIS_TITLE_FONT, standoff=5),
        tickfont=AXIS_TICK_FONT,
        # type="log",
        zeroline=True,
    ),
    legend=dict(
        orientation="h", bgcolor='rgba(0,0,0,0)',
        x=0.5, xanchor="center", y=0.96, yanchor="bottom",
    ),
    margin=dict(t=50, b=10, l=10, r=10, pad=0),
)
citation_counts_fig